# Train/Test Split Metrics

This notebook summarizes:
- which subjects read each story
- which stories are the most read
- which subjects read the most stories
- which 10 stories are the longest in `texts/`

The goal is to keep story and subject coverage visible while designing train/test splits.

In [2]:
from collections import defaultdict
from pathlib import Path

import pandas as pd

RAW_PATH = Path('data/raw')
TEXTS_PATH = Path('texts')
SKIP_MAT_FILES = {'metadata.mat', 'Test.mat'}
SKIP_TEXT_FILES = {'Test'}


def collect_story_subjects(raw_path: Path):
    story_subjects = defaultdict(list)
    subject_stories = {}

    for subject_dir in sorted(path for path in raw_path.iterdir() if path.is_dir()):
        stories = sorted(
            story_file.stem
            for story_file in subject_dir.glob('*.mat')
            if story_file.name not in SKIP_MAT_FILES
        )
        subject_stories[subject_dir.name] = stories

        for story in stories:
            story_subjects[story].append(subject_dir.name)

    return dict(sorted(story_subjects.items())), dict(sorted(subject_stories.items()))


def build_story_summary(story_subjects):
    rows = []
    for story, subjects in story_subjects.items():
        rows.append(
            {
                'story': story,
                'n_subjects': len(subjects),
                'subjects': ', '.join(subjects),
            }
        )
    return pd.DataFrame(rows).sort_values(['n_subjects', 'story'], ascending=[False, True]).reset_index(drop=True)


def build_subject_summary(subject_stories):
    rows = []
    for subject, stories in subject_stories.items():
        rows.append(
            {
                'subject': subject,
                'n_stories': len(stories),
                'stories': ', '.join(stories),
            }
        )
    return pd.DataFrame(rows).sort_values(['n_stories', 'subject'], ascending=[False, True]).reset_index(drop=True)


def build_length_summary(texts_path: Path):
    rows = []
    for text_path in sorted(path for path in texts_path.iterdir() if path.is_file() and path.name not in SKIP_TEXT_FILES):
        text = text_path.read_text(encoding='utf-8')
        rows.append(
            {
                'story': text_path.name,
                'n_words': len(text.split()),
                'n_characters': len(text),
                'n_lines': len(text.splitlines()),
            }
        )
    return pd.DataFrame(rows).sort_values(['n_words', 'story'], ascending=[False, True]).reset_index(drop=True)


story_subjects, subject_stories = collect_story_subjects(RAW_PATH)
story_summary = build_story_summary(story_subjects)
subject_summary = build_subject_summary(subject_stories)
length_summary = build_length_summary(TEXTS_PATH)

coverage_with_length = story_summary.merge(length_summary, on='story', how='left')
coverage_with_length = coverage_with_length.sort_values(['n_subjects', 'n_words', 'story'], ascending=[False, False, True]).reset_index(drop=True)

In [3]:
story_summary.to_csv('story_summary.csv', index=False)

In [4]:
overview = pd.DataFrame(
    {
        'metric': [
            'unique stories in data/raw',
            'subjects in data/raw',
            'stories in texts/',
            'max subjects per story',
            'min subjects per story',
            'max stories per subject',
            'min stories per subject',
        ],
        'value': [
            len(story_subjects),
            len(subject_stories),
            len(length_summary),
            int(story_summary['n_subjects'].max()),
            int(story_summary['n_subjects'].min()),
            int(subject_summary['n_stories'].max()),
            int(subject_summary['n_stories'].min()),
        ],
    }
)

overview

,metric,value
0,unique stories in data/raw,30
1,subjects in data/raw,113
2,stories in texts/,30
3,max subjects per story,61
4,min subjects per story,8
5,max stories per subject,20
6,min stories per subject,2


## Full Story -> Subjects List

In [13]:
pd.set_option('display.max_colwidth', None)
all_subjects_str_long = list(set(', '.join(story_summary.sort_values(by='n_subjects', ascending=True).head(10)['subjects']).split(', ')))
all_subjects_str_short = list(set(', '.join(story_summary.sort_values(by='n_subjects', ascending=False).head(20)['subjects']).split(', ')))

In [20]:
all_subjects_str_short

['sub-095',
 'sub-079',
 'sub-080',
 'sub-098',
 'sub-003',
 'sub-070',
 'sub-086',
 'sub-041',
 'sub-083',
 'sub-006',
 'sub-094',
 'sub-028',
 'sub-082',
 'sub-011',
 'sub-001',
 'sub-109',
 'sub-036',
 'sub-042',
 'sub-027',
 'sub-048',
 'sub-021',
 'sub-058',
 'sub-005',
 'sub-067',
 'sub-002',
 'sub-103',
 'sub-068',
 'sub-056',
 'sub-097',
 'sub-089',
 'sub-075',
 'sub-090',
 'sub-087',
 'sub-030',
 'sub-044',
 'sub-039',
 'sub-009',
 'sub-054',
 'sub-112',
 'sub-013',
 'sub-052',
 'sub-076',
 'sub-101',
 'sub-017',
 'sub-065',
 'sub-022',
 'sub-088',
 'sub-100',
 'sub-105',
 'sub-081',
 'sub-018',
 'sub-093',
 'sub-007',
 'sub-053',
 'sub-055',
 'sub-078',
 'sub-091',
 'sub-012',
 'sub-111',
 'sub-035',
 'sub-071',
 'sub-034',
 'sub-010',
 'sub-016',
 'sub-060',
 'sub-066',
 'sub-073',
 'sub-106',
 'sub-077',
 'sub-085',
 'sub-008',
 'sub-107',
 'sub-046',
 'sub-024',
 'sub-096',
 'sub-064']

In [18]:
required_stories = {
    "La lluvia de fuego",
    "Una rosa para Emilia",
    "Embarrar la magia",
    "Las fotografías",
}

filtered_subjects = (
    subject_summary.assign(
        stories_list=subject_summary["stories"].str.split(", ")
    )
    .loc[lambda df: df["stories_list"].apply(lambda s: required_stories.issubset(set(s)))]
    .sort_values(["n_stories", "subject"], ascending=[False, True])
)

random_24 = filtered_subjects.sample(n=24, random_state=42)

random_24[["subject", "n_stories", "stories"]]

for subj in random_24["subject"]:
    print(f"- {subj}")


- sub-087
- sub-054
- sub-085
- sub-077
- sub-060
- sub-097
- sub-070
- sub-053
- sub-036
- sub-065
- sub-039
- sub-111
- sub-003
- sub-013
- sub-066
- sub-091
- sub-018
- sub-052
- sub-006
- sub-009
- sub-109
- sub-010
- sub-103
- sub-083


## Most Read Stories

In [6]:
story_summary[['story', 'n_subjects']].head(15)

,story,n_subjects
0,El almohadón de plumas,61
1,La lluvia de fuego,61
2,La salud de los enfermos,61
3,Wakefield,61
4,El espejo,60
5,La canción que cantábamos todos los días,60
6,La gallina degollada,60
7,Rubí y el lago danzante,60
8,Una rosa para Emilia,60
9,"Ahora debería reírme, si no estuviera muerto",59


## Subjects Who Read the Most

In [7]:
subject_summary[['subject', 'n_stories']].head(50)

,subject,n_stories
0,sub-003,20
1,sub-006,20
2,sub-009,20
3,sub-010,20
4,sub-013,20
5,sub-018,20
6,sub-027,20
7,sub-034,20
8,sub-036,20
9,sub-039,20


## Coverage Distributions

In [8]:
story_coverage_distribution = (
    story_summary['n_subjects']
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis('subjects_per_story')
    .reset_index(name='n_stories')
)

subject_coverage_distribution = (
    subject_summary['n_stories']
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis('stories_per_subject')
    .reset_index(name='n_subjects')
)

story_coverage_distribution, subject_coverage_distribution

(    subjects_per_story  n_stories
 0                   61          4
 1                   60          5
 2                   59          1
 3                   54          3
 4                   53          3
 5                   52          1
 6                   50          3
 7                   19          1
 8                   16          1
 9                   14          1
 10                  13          2
 11                  12          1
 12                  11          1
 13                   9          2
 14                   8          1,
     stories_per_subject  n_subjects
 0                    20          35
 1                    17           2
 2                    16           1
 3                    12           1
 4                    10          34
 5                     9           1
 6                     8           1
 7                     7           1
 8                     4          15
 9                     3          20
 10                    2        

## 10 Longest Stories in `texts/`

Assumption: "longer" means more words.

In [9]:
top_10_longest = length_summary[['story', 'n_words', 'n_characters', 'n_lines']].head(15)
top_10_longest

,story,n_words,n_characters,n_lines
0,El origen de las especies,4640,29032,1
1,El negro de París,4440,24143,1
2,Sombras sobre vidrio esmerilado 1,3793,21116,1
3,El loco cansino,3265,18718,1
4,Carta a una señorita en París,3182,17681,1
5,Rebeca,2989,18252,1
6,Bienvenido Bob,2964,16417,1
7,Sombras sobre vidrio esmerilado 2,2947,16878,1
8,Carta abierta,2925,17728,1
9,Axolotl,1975,11428,1


## Length + Coverage Together

This is useful for split decisions: you can see which stories are both long and widely read.

In [10]:
coverage_with_length[['story', 'n_subjects', 'n_words', 'n_characters']].head(20)

,story,n_subjects,n_words,n_characters
0,Wakefield,61,843,4506
1,La lluvia de fuego,61,767,4237
2,La salud de los enfermos,61,767,4298
3,El almohadón de plumas,61,719,4099
4,La gallina degollada,60,809,4610
5,El espejo,60,783,4226
6,Rubí y el lago danzante,60,771,4336
7,Una rosa para Emilia,60,767,4282
8,La canción que cantábamos todos los días,60,746,4153
9,"Ahora debería reírme, si no estuviera muerto",59,713,3873


## Lowest-Coverage Stories

These are the easiest to lose if you split aggressively by subject and by story.

In [11]:
story_summary[['story', 'n_subjects']].sort_values(['n_subjects', 'story'], ascending=[True, True]).head(15)

,story,n_subjects
29,Carta a una señorita en París,8
27,Sombras sobre vidrio esmerilado 1,9
28,Sombras sobre vidrio esmerilado 2,9
26,El loco cansino,11
25,Axolotl,12
23,El negro de París,13
24,El origen de las especies,13
22,Carta abierta,14
21,Bienvenido Bob,16
20,Rebeca,19


## Lowest-Coverage Subjects

These subjects contribute the least story diversity.

In [12]:
subject_summary[['subject', 'n_stories']].sort_values(['n_stories', 'subject'], ascending=[False, False])

,subject,n_stories
34,sub-112,20
33,sub-111,20
32,sub-109,20
31,sub-105,20
30,sub-103,20
...,...,...
93,sub-023,3
92,sub-014,3
91,sub-004,3
112,sub-074,2


In [13]:
selected_stories = ['Bienvenido Bob', 'El origen de las especies', 'La lluvia de fuego', 'Una rosa para Emilia', 'Embarrar la magia', 'Las fotografías']
filtered_story_summary = story_summary[story_summary['story'].isin(selected_stories)]
filtered_story_summary

,story,n_subjects,subjects
1,La lluvia de fuego,61,"sub-001, sub-002, sub-003, sub-005, sub-006, sub-009, sub-010, sub-012, sub-013, sub-016, sub-018, sub-021, sub-024, sub-027, sub-028, sub-034, sub-036, sub-039, sub-041, sub-042, sub-044, sub-046, sub-048, sub-052, sub-053, sub-054, sub-058, sub-060, sub-064, sub-065, sub-066, sub-067, sub-068, sub-070, sub-073, sub-075, sub-076, sub-077, sub-078, sub-079, sub-082, sub-083, sub-085, sub-086, sub-087, sub-088, sub-089, sub-091, sub-093, sub-094, sub-095, sub-096, sub-097, sub-098, sub-101, sub-103, sub-105, sub-107, sub-109, sub-111, sub-112"
8,Una rosa para Emilia,60,"sub-001, sub-002, sub-003, sub-005, sub-006, sub-009, sub-010, sub-012, sub-013, sub-016, sub-018, sub-021, sub-024, sub-027, sub-028, sub-034, sub-036, sub-039, sub-041, sub-042, sub-044, sub-046, sub-048, sub-052, sub-053, sub-054, sub-058, sub-060, sub-064, sub-065, sub-066, sub-067, sub-068, sub-070, sub-073, sub-075, sub-076, sub-077, sub-078, sub-079, sub-082, sub-083, sub-085, sub-086, sub-087, sub-088, sub-089, sub-091, sub-093, sub-094, sub-095, sub-096, sub-097, sub-098, sub-103, sub-105, sub-107, sub-109, sub-111, sub-112"
13,Embarrar la magia,53,"sub-003, sub-006, sub-007, sub-008, sub-009, sub-010, sub-011, sub-013, sub-017, sub-018, sub-021, sub-022, sub-027, sub-030, sub-034, sub-035, sub-036, sub-039, sub-048, sub-052, sub-053, sub-054, sub-055, sub-056, sub-058, sub-060, sub-065, sub-066, sub-068, sub-070, sub-071, sub-073, sub-075, sub-077, sub-078, sub-080, sub-081, sub-083, sub-085, sub-086, sub-087, sub-090, sub-091, sub-095, sub-096, sub-097, sub-100, sub-103, sub-105, sub-106, sub-109, sub-111, sub-112"
19,Las fotografías,50,"sub-003, sub-006, sub-007, sub-008, sub-009, sub-010, sub-011, sub-013, sub-017, sub-018, sub-022, sub-027, sub-030, sub-034, sub-035, sub-036, sub-039, sub-048, sub-052, sub-053, sub-054, sub-055, sub-056, sub-058, sub-060, sub-065, sub-066, sub-068, sub-070, sub-071, sub-073, sub-077, sub-078, sub-080, sub-081, sub-083, sub-085, sub-086, sub-087, sub-090, sub-091, sub-096, sub-097, sub-100, sub-103, sub-105, sub-106, sub-109, sub-111, sub-112"
21,Bienvenido Bob,16,"sub-020, sub-023, sub-025, sub-029, sub-032, sub-049, sub-050, sub-051, sub-059, sub-061, sub-069, sub-072, sub-084, sub-092, sub-099, sub-113"
24,El origen de las especies,13,"sub-004, sub-014, sub-015, sub-023, sub-037, sub-043, sub-045, sub-057, sub-059, sub-061, sub-099, sub-108, sub-110"


In [14]:
set_bob = set(story_subjects['Bienvenido Bob'])
set_origen = set(story_subjects['El origen de las especies'])
subjects_in_both = set_bob & set_origen
total_subjects = set_bob | set_origen
set_origen - set_bob

{'sub-004',
 'sub-014',
 'sub-015',
 'sub-037',
 'sub-043',
 'sub-045',
 'sub-057',
 'sub-108',
 'sub-110'}

In [15]:
stories_to_use = {
    row.story: set(row.subjects.split(', '))
    for row in filtered_story_summary.itertuples(index=False)
    if row.story not in {'Bienvenido Bob', 'El origen de las especies'}
}

all_subjects = set().union(*stories_to_use.values())
subjects_in_all_stories = set.intersection(*stories_to_use.values())

subjects_only_in_one_story = {}
for story, subjects in stories_to_use.items():
    other_subjects = set().union(
        *(other_subjects for other_story, other_subjects in stories_to_use.items() if other_story != story)
    )
    subjects_only_in_one_story[story] = subjects - other_subjects

print(f"Number of unique subjects across all selected stories: {len(all_subjects)}")
print("\nSubjects who read all selected stories:")
print(sorted(subjects_in_all_stories))

print("\nSubjects who only read each story:")
for story in sorted(subjects_only_in_one_story):
    only_subjects = sorted(subjects_only_in_one_story[story])
    print(f"{story} ({len(only_subjects)}): {only_subjects}")


Number of unique subjects across all selected stories: 76

Subjects who read all selected stories:
['sub-003', 'sub-006', 'sub-009', 'sub-010', 'sub-013', 'sub-018', 'sub-027', 'sub-034', 'sub-036', 'sub-039', 'sub-048', 'sub-052', 'sub-053', 'sub-054', 'sub-058', 'sub-060', 'sub-065', 'sub-066', 'sub-068', 'sub-070', 'sub-073', 'sub-077', 'sub-078', 'sub-083', 'sub-085', 'sub-086', 'sub-087', 'sub-091', 'sub-096', 'sub-097', 'sub-103', 'sub-105', 'sub-109', 'sub-111', 'sub-112']

Subjects who only read each story:
Embarrar la magia (0): []
La lluvia de fuego (1): ['sub-101']
Las fotografías (0): []
Una rosa para Emilia (0): []


In [16]:
len(subjects_in_all_stories)

35

In [17]:
sorted(list(subjects_in_all_stories))[:7]


['sub-003', 'sub-006', 'sub-009', 'sub-010', 'sub-013', 'sub-018', 'sub-027']